In [62]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks

plt.style.use(r"C:\UserData\z003zh2j\OneDrive - Siemens AG\Documents\plot\matlab.mplstyle")

UR = np.genfromtxt(r"S:\1.FILE\ern\ostroj-ern-abr.csv", delimiter=',', skip_header=0)

t = UR[: , 0]/1e6
a = UR[: , 1]/1e3
b = UR[: , 2]/1e3
#r = UR[: , 3]/1e3

In [ ]:
vysledek = ""
def test(signal):
    global vysledek

    if np.mean(signal) >= (0.05 * np.max(signal)):
        vysledek=("KO! DC offset > 5% Amp")
    else:
        vysledek=("DC offset OK")

test(a)
print("sin A -", vysledek)
test(b)
print("sin B -", vysledek)

polomer = np.sqrt(a**2+b**2)

#print(np.min(polomer),np.max(polomer))

if np.min(polomer) <= 0.75/2 or np.max(polomer) >= 1.2/2:
    print("KO! amplituda")
else:
    print("OK? amplituda")
# pridej tuhle podminku do testovaci funkce

da = np.gradient(a)
db = np.gradient(b)

# Spočítáme úhly pro a a b samostatně s korekcí na kvadrant
uhel_a = np.arcsin(a / polomer)
uhel_b = np.arcsin(b / polomer)

# Pokud signál klesá (derivace < 0), musíme úhel zrcadlit kolem pi/2
uhel_a = np.where(da < 0, np.pi - uhel_a, uhel_a)
uhel_b = np.where(db < 0, np.pi - uhel_b, uhel_b)

# Pokud se dostaneme do záporných hodnot, zarovnáme do rozsahu 0 až 2*pi
uhel_a = np.mod(uhel_a, 2 * np.pi)
uhel_b = np.mod(uhel_b, 2 * np.pi)

# Výsledný fázový rozdíl
fi = uhel_a - uhel_b

# Korekce, aby byl rozdíl vždy v rozumném rozsahu a dělal rovnou čáru
fi = (fi + np.pi) % (2 * np.pi) - np.pi
fi = abs(180/np.pi*fi)
print("mean fázového posunu",np.mean(fi))

plt.figure()
plt.plot(t,polomer)
plt.hlines(0.75/2,0,t[-1]//5,color = 'g',lw = 2)
plt.hlines(1.2/2,0,t[-1]//5,color = 'g', lw = 2)
plt.xlim(0,t[-1]//5)
import matplotlib.patches as patches
obdelnik = patches.Rectangle((0, 0.75/2),t[-1]//5 ,1.2/2-0.75/2 , edgecolor='none', facecolor='lime', alpha=0.25, linewidth=1)
os = plt.gca()
os.add_patch(obdelnik)
plt.show()

plt.figure()
plt.plot(t[0:300],fi[0:300])
plt.plot(t[0:300],fi[3000:3300])
plt.plot(t[0:300],fi[30000:30300])
#plt.xlim(0,0.1)
plt.show()

In [ ]:
da = np.gradient(a)
db = np.gradient(b)

# Spočítáme úhly pro y a y2 samostatně s korekcí na kvadrant
uhel_a = np.arcsin(a / polomer)
uhel_b = np.arcsin(b / polomer)

# Pokud signál klesá (derivace < 0), musíme úhel zrcadlit kolem pi/2
uhel_a = np.where(da < 0, np.pi - uhel_a, uhel_a)
uhel_b = np.where(db < 0, np.pi - uhel_b, uhel_b)

# Pokud se dostaneme do záporných hodnot, zarovnáme do rozsahu 0 až 2*pi
uhel_a = np.mod(uhel_a, 2 * np.pi)
uhel_b = np.mod(uhel_b, 2 * np.pi)

# Výsledný fázový rozdíl
fi = uhel_a - uhel_b

# Korekce, aby byl rozdíl vždy v rozumném rozsahu a dělal rovnou čáru
fi = (fi + np.pi) % (2 * np.pi) - np.pi

# Vykreslení
plt.figure(figsize=(10, 5))
plt.plot(t, a, label='sin A')
plt.plot(t, b, label='sin B')
plt.plot(t, np.abs(fi), label='Fázový rozdíl přes arcsin', color='red', linewidth=2)
plt.xlim(0, 0.2)
plt.ylim(-2, 2)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# --- KLASICKÝ VÝPOČET KLOUZAVÉHO RMS ---
velikost_okna = 83  # Počet vzorků na jednu periodu

import pandas as pd
import numpy as np

def movmean(data, window_size):
    """
    Vypočítá centrovaný klouzavý průměr přesně tak, jak to dělá MATLAB.
    Na okrajích nebere nuly, ale průměruje jen dostupná data.
    """
    # Vytvoříme Pandas Series z dat
    s = pd.Series(data)
    
    # center=True zajistí centrované okno (jako MATLAB)
    # min_periods=1 zajistí, že na okrajích stačí k výpočtu i jen 1 vzorek a nedoplní se nuly
    result = s.rolling(window=window_size, center=True, min_periods=1).mean()
    
    # Vrátíme zpět jako NumPy pole
    return result.to_numpy()

y1=a
y2=b

# RMS = sqrt( průměr( signál^2 ) )
rms1 = np.sqrt(movmean(y1**2, velikost_okna))
rms2 = np.sqrt(movmean(y2**2, velikost_okna))

# Výpočet procentuálního rozdílu v každém bodě
procentualni_rozdil = np.abs(rms1 - rms2) / rms1

# OŘÍZNUTÍ OKRAJŮ: Vynecháme první a poslední část okna
# Pozor: Python indexuje od 0 a horní mez je exkluzivní
okraj = velikost_okna
zkraceny_rozdil = procentualni_rozdil[okraj : -okraj]

# Kontrola tolerance až na oříznutém signálu
je_v_toleranci = np.all(zkraceny_rozdil <= 0.1)

# Vyhledání indexů, kde y2 překračuje mez
H = np.where(y2 > (rms1 * np.sqrt(2) * 1.1))[0]

# Vyhodnocení signálu A
T = rms1 * np.sqrt(2)
if np.all(T > 0.35) and np.all(T < 0.6):
    print(T[83])
    print('A OK')
else:
    print(T[83])
    print('Signály A NEJSOU OK')

# Vyhodnocení signálu B
T2 = rms2 * np.sqrt(2)
if np.all(T2 > 0.35) and np.all(T2 < 0.6):
    print(T2[83])
    print('B OK')
else:
    print(T2[83])
    print('Signály B NEJSOU OK')

# Výpis celkové tolerance
if je_v_toleranci:
    print('Signály jsou v toleranci ±10 % (okraje ignorovány).')
else:
    print('Signály NEJSOU v toleranci ±10 %.')

# --- VYKRESLENÍ GRAFU ---
plt.figure(figsize=(13, 7))  # Velikost v palcích (přibližně odpovídá vašemu poměru)

plt.plot(t, abs(y1), label='Signál y1')
plt.plot(t, abs(y2), label='Signál y2')
plt.plot(t, rms1 * np.sqrt(2), 'r--', linewidth=2, label='Amplituda A')
plt.plot(t, rms1 * np.sqrt(2) * 1.1, 'g--', label='Tolerance +10%')
plt.plot(t, rms1 * np.sqrt(2) * 0.9, 'g--', label='Tolerance -10%')

# Pokud byly nalezeny nějaké body překračující mez, vykreslíme je
if len(H) > 0:
    plt.scatter(t[H], y2[H], color='orange', zorder=5, label='Překročení meze')

plt.title('Kontrola měnící se amplitudy v čase')
plt.legend()
plt.xlim(0, 0.2)
#plt.xlim(3, 3.2)
plt.show()

In [ ]:
def test(signal):
    global vysledek

    if np.mean(signal) >= (0.05 * np.max(signal)):
        vysledek=("KO! DC offset > 5% Amp")
    else:
        vysledek=("DC offset OK")

test(a)
print("sin A -", vysledek)
test(b)
print("sin B -", vysledek)
print(np.mean(a))
print(np.mean(b))

In [ ]:
c = np.linspace(1,499,499)
v = []
for i in range(1,500,1):
    print(i,end="\r")
    rms1 = np.sqrt(movmean(y1**2, i))
    T = rms1 * np.sqrt(2)
    v.append(np.max(T) - np.min(T))
    #print(i, np.max(T) - np.min(T))
plt.plot(c,v)

In [ ]:
import pandas as pd

# Předpokládejme, že 'okno' odpovídá délce jedné periody (např. 100 vzorků)
velikost_okna = 83 

# 1. Změříme kolísavý offset (vyhladíme sinusovku)
kolisavy_offset = pd.Series(a).rolling(window=velikost_okna, center=True).mean()

# 2. Odečteme ho od původního signálu
vycisteny_signal = a - kolisavy_offset

plt.figure(figsize=(10, 5))
plt.plot(t,kolisavy_offset)
plt.xlim(0, 0.2)

In [ ]:
radius = np.sqrt(a**2 + b**2)
plt.plot(t, radius, label='Vypočítaná amplitúda $\\sqrt{A^2 + B^2}$', color='purple')
#plt.axhline(y=20000, color='r', linestyle='--', label='Minimálny limit pre chybu (~20000)')
plt.axhline(y=0.5, color='g', linestyle=':', label='Nominálna úroveň (0.5)')
plt.xlim(0,100)
plt.ylim(0,1)

In [64]:
import numpy as np

# --- 1. Simulace tvých dat (pro ukázku) ---
# Předpokládejme vzorkování a dvě sinusovky s offsetem a šumem
t = np.linspace(0, 10, 1000)
# Teoretická amplituda 0.5V znamená Vpp = 1.0V
signal_A = a
signal_B = b

# --- 2. Odstranění DC Offsetu a výpočet Vpp ---
# Pro reálná kolísající data je lepší počítat Vpp z lokálních extrémů
# Pokud analyzuješ celý balík dat naráz, stačí global max - min:

def analyze_signals(sig_a, sig_b):
    # Odstranění DC složky pro přesnější analýzu (volitelné, pro Vpp to de facto netřeba, ale dobrá praxe)
    sig_a_detrend = sig_a - np.mean(sig_a)
    sig_b_detrend = sig_b - np.mean(sig_b)
    
    # Výpočet Peak-to-Peak (Vpp)
    vpp_a = np.ptp(sig_a)  # np.ptp je zkratka pro peak-to-peak (max - min)
    vpp_b = np.ptp(sig_b)
    
    # --- 3. Vyhodnocení podmínek ---
    
    # Podmínka 1: Vpp je v rozsahu 0.75V až 1.2V
    lim_min, lim_max = 0.75, 1.2
    cond_vpp_a = lim_min <= vpp_a <= lim_max
    cond_vpp_b = lim_min <= vpp_b <= lim_max
    
    # Podmínka 2: A = B +- 10%
    # Bere se to většinou vůči hodnotě A, nebo vůči průměru obou. Vezměme vůči A:
    relative_difference = abs(vpp_a - vpp_b) / vpp_a
    cond_amplitude_match = relative_difference <= 0.10
    
    # Výpis výsledků
    print(f"Vpp Signálu A: {vpp_a:.3f} V (V normě: {cond_vpp_a})")
    print(f"Vpp Signálu B: {vpp_b:.3f} V (V normě: {cond_vpp_b})")
    print(f"Relativní rozdíl amplitud: {relative_difference * 100:.2f}% (V normě: {cond_amplitude_match})")
    
    success = cond_vpp_a and cond_vpp_b and cond_amplitude_match
    return success

# Spuštění funkce
vysledek = analyze_signals(signal_A, signal_B)
print(f"\nSplňují signály celkově podmínky? -> {vysledek}")

Vpp Signálu A: 1.096 V (V normě: True)
Vpp Signálu B: 1.045 V (V normě: True)
Relativní rozdíl amplitud: 4.62% (V normě: True)

Splňují signály celkově podmínky? -> True


In [75]:
def analyze_signals(sig_a, sig_b):
    
    # Příklad rozsekání na okna o velikosti např. 100 vzorků
    window_size = 83
    for i in range(0, len(sig_a), window_size):
        chunk_a = sig_a[i:i+window_size]
        chunk_b = sig_b[i:i+window_size]
        if len(chunk_a) < window_size: break # Konec dat
        
        vpp_a = np.ptp(chunk_a)
        vpp_b = np.ptp(chunk_b)
        # A tady provedeš stejné if/else podmínky jako v předchozím kódu...
    
    # --- 3. Vyhodnocení podmínek ---
    
    # Podmínka 1: Vpp je v rozsahu 0.75V až 1.2V
    lim_min, lim_max = 0.75, 1.2
    cond_vpp_a = lim_min <= vpp_a <= lim_max
    cond_vpp_b = lim_min <= vpp_b <= lim_max
    
    # Podmínka 2: A = B +- 10%
    # Bere se to většinou vůči hodnotě A, nebo vůči průměru obou. Vezměme vůči A:
    relative_difference = abs(vpp_a - vpp_b) / vpp_a
    cond_amplitude_match = relative_difference <= 0.10
    
    # Výpis výsledků
    print(f"Vpp Signálu A: {vpp_a:.3f} V (V normě: {cond_vpp_a})")
    print(f"Vpp Signálu B: {vpp_b:.3f} V (V normě: {cond_vpp_b})")
    print(f"Relativní rozdíl amplitud: {relative_difference * 100:.2f}% (V normě: {cond_amplitude_match})")
    
    return 

# Spuštění funkce
analyze_signals(a, b)


Vpp Signálu A: 0.979 V (V normě: True)
Vpp Signálu B: 0.919 V (V normě: True)
Relativní rozdíl amplitud: 6.11% (V normě: True)
